In [ ]:
import subprocess
subprocess.run(["pip", "install", "pandas", "anthropic", "sentence-transformers", "faiss-cpu"])

In [ ]:
import pandas as pd
import anthropic
print("Everything is working! Let's build FashionFind 🚀")

In [ ]:
import pandas as pd

# Our fashion product catalog
data = {
    "product_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "name": [
        "Floral Kurta",
        "Slim Fit Jeans",
        "Embroidered Anarkali",
        "Casual White T-Shirt",
        "Silk Saree",
        "Denim Jacket",
        "Printed Maxi Dress",
        "Palazzo Pants",
        "Woolen Sweater",
        "Ethnic Lehenga"
    ],
    "category": [
        "Kurta", "Jeans", "Kurta", "T-Shirt", "Saree",
        "Jacket", "Dress", "Pants", "Sweater", "Lehenga"
    ],
    "price": [999, 1499, 2999, 499, 4999, 1999, 1799, 899, 1299, 5999],
    "color": [
        "Pink", "Blue", "Red", "White", "Golden",
        "Blue", "Multicolor", "Black", "Grey", "Pink"
    ],
    "description": [
        "A beautiful pink floral kurta perfect for casual outings and festivals",
        "Classic slim fit blue jeans suitable for everyday casual wear",
        "Elegant red embroidered anarkali suit perfect for weddings and parties",
        "Simple and comfortable white t-shirt for daily casual wear",
        "Gorgeous golden silk saree ideal for weddings and special occasions",
        "Trendy blue denim jacket perfect for winter casual outings",
        "Vibrant multicolor printed maxi dress great for beach and vacations",
        "Comfortable black palazzo pants suitable for office and casual wear",
        "Warm grey woolen sweater perfect for cold weather and winters",
        "Stunning pink ethnic lehenga choli perfect for weddings and festivals"
    ]
}

df = pd.DataFrame(data)
print(f"Dataset created with {len(df)} products!")
print(df[["name", "price", "color"]].to_string())

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert all product descriptions into embeddings
print("Converting product descriptions to embeddings...")
embeddings = model.encode(df['description'].tolist())

print(f"✅ Done! Each product is now represented as {embeddings.shape[1]} numbers")
print(f"Total products embedded: {embeddings.shape[0]}")

In [ ]:
import faiss
import numpy as np

# Convert embeddings to the right format
embedding_matrix = np.array(embeddings).astype('float32')

# Build the FAISS index (our searchable vector store)
dimension = embedding_matrix.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print(f"✅ Vector store created with {index.ntotal} products indexed!")

# Test it — search for products similar to a query
query = "I want something for a wedding"
query_embedding = model.encode([query]).astype('float32')

# Find top 3 most similar products
distances, indices = index.search(query_embedding, k=3)

print(f"\n🔍 Query: '{query}'")
print("\nTop 3 matching products:")
for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {df['name'][idx]} — ₹{df['price'][idx]} ({df['color'][idx]})")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "groq"])

In [ ]:
from groq import Groq

# Initialize Groq client
client = Groq(api_key="YOUR_GROQ_API_KEY")

def fashion_assistant(user_query, budget=None):
    # Step 1: Search vector store
    query_embedding = model.encode([user_query]).astype('float32')
    distances, indices = index.search(query_embedding, k=3)
    
    # Step 2: Get matching products
    results = []
    for idx in indices[0]:
        product = df.iloc[idx]
        if budget is None or product['price'] <= budget:
            results.append(f"- {product['name']} in {product['color']}: ₹{product['price']} — {product['description']}")
    
    context = "\n".join(results)
    
    # Step 3: Ask Groq to generate a response
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": f"""You are a helpful Myntra fashion assistant.
                
A customer asked: "{user_query}"

Here are the relevant products from our catalog:
{context}

Give a friendly, helpful recommendation in 3-4 sentences."""
            }
        ]
    )
    
    return response.choices[0].message.content

# Test it!
response = fashion_assistant("I need something for a wedding", budget=5000)
print(response)

In [ ]:
# Test 1 — budget filter
print(fashion_assistant("I want a casual outfit", budget=1000))

In [ ]:
# Test 2 — color preference
print(fashion_assistant("Show me something in pink"))

In [ ]:
# Test 3 — weather based
print(fashion_assistant("It's getting cold, what should I wear?"))

In [ ]:
print("""
🚀 FashionFind — AI Fashion Assistant
=====================================
Built by: Neeta Singh

Tech Stack:
- Sentence Transformers (embeddings)
- FAISS (vector search)
- Groq LLaMA 3.3 (LLM)
- Pandas (data pipeline)

How it works:
1. Product descriptions → 384-dimensional embeddings
2. User query → embedding → FAISS similarity search
3. Top matching products → LLM → natural response

This is a RAG (Retrieval Augmented Generation) pipeline.
""")